In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.coffee_secure
COMMENT 'Personal data. No grants to analysts';

CREATE OR REPLACE TABLE workspace.coffee_secure.customers (
  customer_id INT,
  name STRING,
  email STRING,
  phone STRING,
  city STRING
);

INSERT INTO workspace.coffee_secure.customers VALUES
  (10, 'Lena Fischer', 'lena.fischer@example.com', '+49 30 5550 1010', 'Berlin'),
  (11, 'Jonas Weber', 'jonas.weber@example.com', '+49 40 5550 1011', 'Hamburg'),
  (12, 'Aylin Demir', 'aylin.demir@example.com', '+49 30 5550 1012', 'Berlin'),
  (13, 'Tom Becker', 'tom.becker@example.com', '+49 40 5550 1013', 'Hamburg'),
  (14, 'Sara Novak', 'sara.novak@example.com', '+49 30 5550 1014', 'Berlin'),
  (15, 'Max Hoffmann', 'max.hoffmann@example.com', '+49 40 5550 1015', 'Hamburg'),
  (16, 'Mina Rahimi', 'mina.rahimi@example.com', '+49 30 5550 1016', 'Berlin');

SELECT * FROM workspace.coffee_secure.customers ORDER BY customer_id;

In [0]:
SELECT *
FROM workspace.information_schema.row_filters
WHERE table_schema = 'coffee_secure'
  AND table_name = 'customers';

In [0]:
CREATE OR REPLACE FUNCTION workspace.coffee_secure.mask_email(email STRING)
RETURNS STRING
RETURN CASE
  WHEN current_user() = 'mia@example.com' THEN email
  ELSE concat('***@', split_part(email, '@', 2))
END;

In [0]:
ALTER TABLE workspace.coffee_secure.customers
  ALTER COLUMN email SET MASK workspace.coffee_secure.mask_email;

SELECT customer_id, name, email FROM workspace.coffee_secure.customers
ORDER BY customer_id;

In [0]:
CREATE OR REPLACE FUNCTION workspace.coffee_secure.mask_email(email STRING)
RETURNS STRING
RETURN CASE
  WHEN current_user() IN ('mia@example.com', 'you@example.com') THEN email
  ELSE concat('***@', split_part(email, '@', 2))
END;

In [0]:
ALTER TABLE workspace.coffee_secure.customers
  ALTER COLUMN email SET MASK workspace.coffee_secure.mask_email;

SELECT customer_id, name, email FROM workspace.coffee_secure.customers
ORDER BY customer_id;

In [0]:
CREATE OR REPLACE FUNCTION workspace.coffee_secure.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE
  WHEN current_user() = 'mia@example.com' THEN phone
  ELSE concat('**** ', right(phone, 4))
END;

ALTER TABLE workspace.coffee_secure.customers
  ALTER COLUMN phone SET MASK workspace.coffee_secure.mask_phone;

In [0]:
SELECT * FROM workspace.coffee_secure.customers ORDER BY customer_id;

In [0]:
CREATE OR REPLACE FUNCTION workspace.coffee_secure.berlin_only(city STRING)
RETURNS BOOLEAN
RETURN current_user() = 'mia@example.com' OR city = 'Berlin';

ALTER TABLE workspace.coffee_secure.customers
  SET ROW FILTER workspace.coffee_secure.berlin_only ON (city);

SELECT customer_id, name, city FROM workspace.coffee_secure.customers
ORDER BY customer_id;

SELECT COUNT(*) AS customers FROM workspace.coffee_secure.customers;

In [0]:
CREATE OR REPLACE VIEW workspace.coffee_reports.customer_revenue AS
SELECT c.customer_id, c.name, c.email, c.phone, c.city,
       COUNT(DISTINCT s.order_id) AS orders,
       ROUND(SUM(s.line_amount_eur), 2) AS revenue_eur
FROM workspace.coffee_secure.customers AS c
JOIN workspace.coffee_pipeline.silver_order_lines AS s
  ON s.customer_id = c.customer_id
GROUP BY c.customer_id, c.name, c.email, c.phone, c.city;

GRANT SELECT ON TABLE workspace.coffee_reports.customer_revenue TO `account users`;















































































In [0]:
SELECT * FROM workspace.coffee_reports.customer_revenue ORDER BY customer_id;

In [0]:
SELECT * FROM workspace.coffee_reports.customer_revenue ORDER BY customer_id;

In [0]:
SELECT * FROM workspace.coffee_reports.customer_revenue ORDER BY customer_id;